In [ ]:
import json
import os
import random
import numpy as np
import pandas as pd
import os
import csv
import json
import shutil

from pydub import AudioSegment
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
#nascodono avvisi non importatni

In [ ]:

def organize_raw_datasets():
    
    # Controllo se split già fatto
    split_marker = "Dataset/audio_sources/.split_done"
    if os.path.exists(split_marker):
        print("Split già completato! Skipping...")
        return
    
    print("Inizio organizzazione dataset...")
    
    # Crea struttura cartelle
    base_splits = ['train', 'test', 'validation']
    categories = ['background', 'event', 'nash']
    
    print("Creazione cartelle...")
    os.makedirs('Dataset/audio_sources/dataset', exist_ok=True)
    
    for split in base_splits:
        for category in categories:
            path = f'Dataset/audio_sources/dataset/{split}/{category}'
            os.makedirs(path, exist_ok=True)
            print(f"    {path}")
    
    # ESC-50: 
    print("ESC-50...")
    
    if not os.path.exists('Dataset/ESC-50-master/meta/esc50.csv'):
        print(" ESC-50 non trovato - skip")
    else:
        esc_df = pd.read_csv('Dataset/ESC-50-master/meta/esc50.csv')
        
        # Definizioni categorie
        animali = [
            'dog', 'cat', 'chirping_birds', 'rooster', 'pig', 'cow', 'frog', 'crickets',  
            'crow', 'hen', 'sheep', 'insects'  
        ]
        
        non_animali = [
            'rain', 'sea_waves', 'crackling_fire', 'wind', 'thunderstorm', 'helicopter', 
            'airplane', 'train', 'car_horn', 'engine', 'footsteps', 'door_wood_knock'   
        ]
        
        # BACKGROUND (non-animali)
        print("  Processando background...")
        esc_background = esc_df[esc_df['category'].isin(non_animali)]
        background_stats = {}
        
        for categoria in non_animali:
            categoria_files = esc_background[esc_background['category'] == categoria]
            file_list = categoria_files['filename'].tolist()
            
            if len(file_list) == 0:
                continue
                
            # Crea sottocartella per categoria
            for split in base_splits:
                os.makedirs(f'Dataset/audio_sources/dataset/{split}/background/{categoria}', exist_ok=True)
            
            # Split 80/10/10 - SEED FISSO 42 
            random.seed(42)
            random.shuffle(file_list)
            
            n = len(file_list)
            train_end = max(1, int(0.8 * n))
            test_end = max(train_end + 1, int(0.9 * n))
            
            train_files = file_list[:train_end]
            test_files = file_list[train_end:test_end]
            val_files = file_list[test_end:]
            
            background_stats[categoria] = {
                'total': n, 'train': len(train_files), 
                'test': len(test_files), 'val': len(val_files)
            }
            
            # Copia file
            for split_name, files, split_folder in [("train", train_files, "train"), 
                                                   ("test", test_files, "test"), 
                                                   ("val", val_files, "validation")]:
                for filename in files:
                    src = f"Dataset/ESC-50-master/audio/{filename}"
                    dst = f"Dataset/audio_sources/dataset/{split_folder}/background/{categoria}/{filename}"
                    if os.path.exists(src):
                        shutil.copy2(src, dst)
            
            print(f"    {categoria}: {n} files")
        
        # EVENT (animali)
        print("  Processando events...")
        esc_animali = esc_df[esc_df['category'].isin(animali)]
        event_stats = {}
        
        for categoria in animali:
            categoria_files = esc_animali[esc_animali['category'] == categoria]
            file_list = categoria_files['filename'].tolist()
            
            if len(file_list) == 0:
                continue
                
            # Crea sottocartelle
            for split in base_splits:
                os.makedirs(f'Dataset/audio_sources/dataset/{split}/event/{categoria}', exist_ok=True)
            
            # Split
            random.seed(42)  # SEED FISSO PER SPLIT CONSISTENTE
            random.shuffle(file_list)
            
            n = len(file_list)
            train_end = max(1, int(0.8 * n))
            test_end = max(train_end + 1, int(0.9 * n))
            
            train_files = file_list[:train_end]
            test_files = file_list[train_end:test_end]
            val_files = file_list[test_end:]
            
            event_stats[categoria] = {
                'total': n, 'train': len(train_files), 
                'test': len(test_files), 'val': len(val_files)
            }
            
            # Copia file
            for split_name, files, split_folder in [("train", train_files, "train"), 
                                                   ("test", test_files, "test"), 
                                                   ("val", val_files, "validation")]:
                for filename in files:
                    src = f"Dataset/ESC-50-master/audio/{filename}"
                    dst = f"Dataset/audio_sources/dataset/{split_folder}/event/{categoria}/{filename}"
                    if os.path.exists(src):
                        shutil.copy2(src, dst)
            
            print(f"    {categoria}: {n} files")
    
    # URBANSOUND8K 
    print("UrbanSound8K")
    
    if not os.path.exists('Dataset/UrbanSound8K/metadata/Dataset/UrbanSound8K.csv'):
        print("  Dataset/UrbanSound8K non trovato - skip")
    else:
        urban_df = pd.read_csv('Dataset/UrbanSound8K/metadata/Dataset/UrbanSound8K.csv')
        
        # Escludi dog_bark 
        excluded_classes = ['dog_bark']
        urban_background = urban_df[~urban_df['class'].isin(excluded_classes)]
        urban_classes = urban_background['class'].unique()
        
        for classe in urban_classes:
            # Crea sottocartelle
            for split in base_splits:
                os.makedirs(f'Dataset/audio_sources/dataset/{split}/background/{classe}', exist_ok=True)
            
            classe_files = urban_background[urban_background['class'] == classe]
            file_info = [(row['fold'], row['slice_file_name']) for _, row in classe_files.iterrows()]
            
            if len(file_info) == 0:
                continue
                
            # Split
            random.seed(42)  # SEED FISSO PER SPLIT CONSISTENTE
            random.shuffle(file_info)
            
            n = len(file_info)
            train_end = max(1, int(0.8 * n))
            test_end = max(train_end + 1, int(0.9 * n))
            
            train_files = file_info[:train_end]
            test_files = file_info[train_end:test_end]
            val_files = file_info[test_end:]
            
            # Copia file
            for split_name, files, split_folder in [("train", train_files, "train"), 
                                                   ("test", test_files, "test"), 
                                                   ("val", val_files, "validation")]:
                for fold, filename in files:
                    src = f"Dataset/UrbanSound8K/audio/fold{fold}/{filename}"
                    dst = f"Dataset/audio_sources/dataset/{split_folder}/background/{classe}/{filename}"
                    if os.path.exists(src):
                        shutil.copy2(src, dst)
            
            print(f"    {classe}: {n} files")
    
    # NORMALTARGZ (NASH)
    print("Normaltargz")
    
    if not os.path.exists("Dataset/Normaltargz"):
        print("  Normaltargz non trovato - skip")
    else:
        # Raccoglie tutti i file
        normal_files = []
        for filename in os.listdir("Dataset/Normaltargz"):
            if filename.lower().endswith(('.wav', '.flac')):
                normal_files.append(filename)
        
        if len(normal_files) > 0:
            # Split 80/10/10
            random.seed(42)  # SEED FISSO PER SPLIT CONSISTENTE
            random.shuffle(normal_files)
            
            n = len(normal_files)
            train_end = max(1, int(0.8 * n))
            test_end = max(train_end + 1, int(0.9 * n))
            
            train_files = normal_files[:train_end]
            test_files = normal_files[train_end:test_end]
            val_files = normal_files[test_end:]
            
            # Copia file (direttamente in nash/, senza sottocartelle)
            for split_name, files, split_folder in [("train", train_files, "train"), 
                                                   ("test", test_files, "test"), 
                                                   ("val", val_files, "validation")]:
                for filename in files:
                    src = f"Dataset/Normaltargz/{filename}"
                    dst = f"Dataset/audio_sources/dataset/{split_folder}/nash/{filename}"
                    if os.path.exists(src):
                        shutil.copy2(src, dst)
            
            print(f"    Nash: {n} files : train={len(train_files)}, test={len(test_files)}, val={len(val_files)}")
    
    # Crea marker per evitare ripetizioni
    with open(split_marker, 'w') as f:
        f.write("Split completed successfully")
    
    print("SPLIT COMPLETATO!")
    print("   Dataset/audio_sources/dataset/train/{background,event,nash}/")
    print("   Dataset/audio_sources/dataset/test/{background,event,nash}/")
    print("   Dataset/audio_sources/dataset/validation/{background,event,nash}/")


In [ ]:
# Controlla se già fatto
split_marker = "Dataset/audio_sources/.split_done"
if os.path.exists(split_marker):
    print(" Split già completato, skip!")
    
    # Mostra cosa c'è
    print("Contenuto attuale:")
    for split in ['train', 'test', 'validation']:
        for category in ['background', 'event', 'nash']:
            path = f'Dataset/audio_sources/dataset/{split}/{category}'
            if os.path.exists(path):
                if category == 'nash':
                    # Nash: file diretti
                    files = [f for f in os.listdir(path) if f.endswith(('.wav', '.flac'))]
                    print(f"   {split}/{category}: {len(files)} files")
                else:
                    # Background/Event: sottocartelle
                    subcats = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
                    total_files = 0
                    for subcat in subcats:
                        subcat_path = os.path.join(path, subcat)
                        files = [f for f in os.listdir(subcat_path) if f.endswith(('.wav', '.flac'))]
                        total_files += len(files)
                    print(f"   {split}/{category}: {len(subcats)} categorie, {total_files} files totali")
            else:
                print(f"   {split}/{category}: NON ESISTE")

else:
    print("⚡ Eseguendo split...")
    organize_raw_datasets()
    print(" Split completato!")


In [ ]:

# Scegli config 
config_file = 'config_validation.json'  # Cambiare json:test/train/validation !!!!
print(f" Usando: {config_file}")

with open(config_file, 'r') as file:
    config = json.load(file)


# Determina quale split usare dal nome del config
if 'train' in config_file:
    split_name = 'train'
elif 'test' in config_file:
    split_name = 'test'
elif 'validation' in config_file:
    split_name = 'validation'
else:
    split_name = 'train'  # default

print(f" Split selezionato: {split_name}")

# Aggiorna i paths
config['paths']['background_segments_path'] = f"Dataset/audio_sources/dataset/{split_name}/background"
config['paths']['event_segments_path'] = f"Dataset/audio_sources/dataset/{split_name}/event"
config['paths']['nash_segments_path'] = f"Dataset/audio_sources/dataset/{split_name}/nash"
config['paths']['output_path'] = f"Dataset/audio_sources/mixtures/{split_name}"


print("Params:")
for key, value in config["params"].items():  
    print(f"  {key}: {value}")

print("Paths:")
for key, value in config["paths"].items():   
    print(f"  {key}: {value}")

# Crea cartelle output
print("Cartelle output...")
for key, path in config["paths"].items():
    if path.endswith(('.csv', '.json')):
        continue
    try:
        os.makedirs(path, exist_ok=True)
        print(f" {path}")
    except Exception as e:
        print(f"  Errore {path}: {e}")

# Setup seed 
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    print(f" Seed impostato: {seed}")


seed_everything(config["params"]["seed"])
print(f" Le mixtures usano seed: {config['params']['seed']}")

# Verifica cartelle input
paths_to_check = [
    config['paths']['background_segments_path'],
    config['paths']['event_segments_path'], 
    config['paths']['nash_segments_path']
]

all_ok = True
for path in paths_to_check:
    if os.path.exists(path):
        if 'nash' in path:
            # Nash: conta file diretti
            files = [f for f in os.listdir(path) if f.endswith(('.wav', '.flac'))]
            print(f"   {path}: {len(files)} files")
        else:
            # Background e Event: conta sottocartelle
            subcats = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
            print(f"   {path}: {len(subcats)} categorie")
    else:
        print(f"   {path}: NON ESISTE!")
        all_ok = False

if all_ok:
    print("Pronto per generare mixtures!")
else:
    print("ATTENZIONE: mancano delle cartelle. Esegui split!")

print(f"Genererò {config['params']['num_mixes']} mixtures in: {config['paths']['output_path']}")


In [ ]:

def get_random_segments(segments_paths_list, num_segments_list):

    audio_extensions = ('.wav', '.flac')
    random_segments = []

    for segments_paths, num_segments in zip(segments_paths_list, num_segments_list):
        for _ in range(num_segments):
            if not os.path.exists(segments_paths):
                raise ValueError(f"Cartella non esistente: {segments_paths}")
            
            items = os.listdir(segments_paths)
            subdirs = [item for item in items if os.path.isdir(os.path.join(segments_paths, item))]
            
            if len(subdirs) > 0:
                # CASO: sottocartelle (background/event)
                selected_class = random.choice(subdirs)
                class_dir = os.path.join(segments_paths, selected_class)
                valid_files = [f for f in os.listdir(class_dir) if f.lower().endswith(audio_extensions)]
                
                if not valid_files:
                    print(f"  Nessun file audio in {class_dir}")
                    continue
                    
                selected_segment = random.choice(valid_files)
                segment_path = os.path.join(class_dir, selected_segment)
            else:
                # CASO: file diretti (nash)
                valid_files = [f for f in items if f.lower().endswith(audio_extensions)]
                
                if not valid_files:
                    raise ValueError(f"Nessun file audio in {segments_paths}")
                
                selected_segment = random.choice(valid_files)
                segment_path = os.path.join(segments_paths, selected_segment)
            
            random_segments.append(segment_path)

    return random_segments

# INIZIO GENERAZIONE MIXTURES
print(f" Generando {config['params']['num_mixes']} mixtures...")

for num_mix in tqdm(range(config["params"]["num_mixes"]), desc="Processing mixes"):
    # Lunghezza mix
    len_mixture = random.randint(config['params']['min_len_mixture'], config['params']['max_len_mixture'])
    
    # Audio base
    mixture_audio = AudioSegment.silent(duration=len_mixture).set_channels(config["params"]["channel"]).set_frame_rate(config["params"]["frame_rate"])

    # BACKGROUND 
    background_audio = AudioSegment.silent(duration=len_mixture).set_channels(config["params"]["channel"]).set_frame_rate(config["params"]["frame_rate"])
    
    # 1. Segmento NASH 
    try:
        nash_segment = get_random_segments(
            [config["paths"]["nash_segments_path"]],
            [1]
        )
    except Exception as e:
        print(f" Errore nash mix {num_mix}: {e}")
        continue
    
    # 2. Segmenti background 
    num_additional_background = random.randint(
        config["params"]["min_num_background_segments"],
        config["params"]["max_num_background_segments"]
    )
    
    try:
        additional_segments = get_random_segments(
            [config["paths"]["background_segments_path"]],
            [num_additional_background]
        )
    except Exception as e:
        print(f" Errore background mix {num_mix}: {e}")
        continue
    
    # Combina tutti i background
    all_background = nash_segment + additional_segments
    
    background_info = []
    for segment_path in all_background:
        try:
            segment_audio = AudioSegment.from_file(segment_path)
            segment_audio = segment_audio.normalize()
            
            # Posizionamento casuale
            max_start = max(0, len_mixture - len(segment_audio))
            start = random.randint(0, max_start) if max_start > 0 else 0
            end = start + len(segment_audio)
            
            # Overlay
            background_audio = background_audio.overlay(segment_audio, position=start)
            
            # Info per JSON
            background_info.append({
                "Path": segment_path,
                "Class": os.path.basename(os.path.dirname(segment_path)) if os.path.dirname(segment_path).split('/')[-1] != 'nash' else 'nash',
                "Start": start,
                "End": end
            })
            
        except Exception as e:
            print(f"  Errore segmento background {segment_path}: {e}")
            continue

    # Overlay background sul mix
    mixture_audio = mixture_audio.overlay(background_audio)

    #  EVENT = versi animali
    event_info = []
    event_audio = AudioSegment.silent(duration=len_mixture).set_channels(config["params"]["channel"]).set_frame_rate(config["params"]["frame_rate"])
    
    include_event = random.random() < 1  # sempre True
    
    if include_event:
        num_event_segments = random.randint(
            config["params"]["min_num_event_segments"], 
            config["params"]["max_num_event_segments"]
        )
        
        try:
            random_events = get_random_segments(
                [config["paths"]["event_segments_path"]], 
                [num_event_segments]
            )
            
            for segment_path in random_events:
                try:
                    segment_audio = AudioSegment.from_file(segment_path)
                    segment_audio = segment_audio.normalize()
                    segment_audio = segment_audio.apply_gain(10)  # Gain per eventi
                    
                    # Posizionamento
                    max_start = max(0, len_mixture - len(segment_audio))
                    start = random.randint(0, max_start) if max_start > 0 else 0
                    end = start + len(segment_audio)
                    
                    # Overlay
                    event_audio = event_audio.overlay(segment_audio, position=start)
                    
                    # Info
                    event_info.append({
                        "Path": segment_path,
                        "Class": os.path.basename(os.path.dirname(segment_path)),
                        "Start": start,
                        "End": end
                    })
                    
                except Exception as e:
                    print(f"  Errore segmento event {segment_path}: {e}")
                    continue
                    
        except Exception as e:
            print(f" Errore eventi mix {num_mix}: {e}")
    
    # Overlay eventi sul mix
    mixture_audio = mixture_audio.overlay(event_audio)

    #  SALVATAGGIO
    mix_dir = os.path.join(config["paths"]["output_path"], f"mix_{num_mix}")
    os.makedirs(mix_dir, exist_ok=True)

    try:
        # Salva i 4 file:
        #1) audio 
        background_audio.export(os.path.join(mix_dir, "background.wav"), format="wav")
        event_audio.export(os.path.join(mix_dir, "event.wav"), format="wav") 
        mixture_audio.export(os.path.join(mix_dir, "mixture.wav"), format="wav")

        #2) JSON con metadati
        mixture_info = {
            "background_info": background_info,
            "event_info": event_info if event_info else None
        }
        with open(os.path.join(mix_dir, "mixture_info.json"), 'w') as f:
            json.dump(mixture_info, f, indent=4)
            
    except Exception as e:
        print(f" Errore salvataggio mix {num_mix}: {e}")
        continue

print("GENERAZIONE COMPLETATA!")
print(f" Mixtures salvate in: {config['paths']['output_path']}")